In [1]:
# 정수론적 변환(NTT) - 모듈러 연산에서의 FFT


# 재귀 FFT
def fft_rec(f: list[int], ws: list[int], p: int) -> list[int]:
    """
    f = 차수 < N인 다항식
    ws[i] = w^i, 여기서 w는 N차 원시 단위근
    p = x mod p
    [f(ws[0]), f(ws[1]), ..., f(ws[N-1])] 반환
    """
    n = len(f)
    assert n & (n - 1) == 0, f"{n} is not a power of 2"
    assert len(ws) == n, f'{len(ws)} != {n}'

    if n == 1:
        return f

    # 선택 사항 - x가 N차 원시 단위근인지 확인
    w = ws[1]
    assert pow(w, n, p) == 1, f"{w}^{n} mod {p} != 1"
    assert pow(w, n // 2, p) == p - 1, f"{w}^({n} / 2) mod {p} != -1"

    f_even = fft_rec(f[::2], ws[::2], p)
    f_odd = fft_rec(f[1::2], ws[::2], p)
    ys = [0] * n

    h = n // 2
    for i in range(h):
        # -1 = w^(n/2)
        # -w^i = -1 * w^i = w^(n/2 + i)
        # f(x)  = f_even(x^2) + x * f_odd(x^2)
        # f(-x) = f_even(x^2) - x * f_odd(x^2)
        ys[i] = (f_even[i] + ws[i] * f_odd[i]) % p
        ys[h + i] = (f_even[i] - ws[i] * f_odd[i]) % p

    return ys


# 재귀 없는 FFT
# N개 점(ws)에서 다항식 f의 값 계산
def fft(f: list[int], ws: list[int], p: int) -> list[int]:
    n = len(f)
    assert n & (n - 1) == 0, f"{n} is not a power of 2"
    assert len(ws) == n, f'{len(ws)} != {n}'

    ys = [0] * n

    # 짝수 항과 홀수 항의 최종 위치를 ys에 대응시킴
    # 비트 순서 반전
    # 시작 인덱스 = 최종 인덱스의 비트 순서를 뒤집은 값
    rev = 0
    for i in range(n):
        ys[i] = f[rev]
        # 왼쪽에서 오른쪽으로 올림 처리
        mask = n >> 1
        while rev & mask:
            # 마스크가 1인 위치를 0으로 설정
            rev &= ~mask
            # 1을 오른쪽으로 이동
            mask >>= 1
        # 올림 처리 후 올바른 비트 위치에 1을 배치
        rev |= mask

    # 병합
    k = n
    s = 2
    while k > 1:
        for i in range(0, n, s):
            h = s // 2
            for j in range(i, i + h):
                f_even = ys[j]
                f_odd = ys[j + h]
                # 반복 k에서 wi = (w^j)^k = w^(j * k % n),
                #      따라서 다음 반복의 wi = w^(j * (k // 2) % n)
                wi = ws[(j * (k // 2)) % n]

                ys[j] = (f_even + wi * f_odd) % p
                ys[j + h] = (f_even - wi * f_odd) % p

        s *= 2
        k //= 2

    return ys


# 역 FFT
# N개 평가값(ys)으로 차수 < N인 다항식을 보간
# 역 FFT = N^(-1) * fft(ys, [1, w^(-1), w^(-2), ..., w^(-(N-1))], p)
def ifft(ys: list[int], ws: list[int], p: int) -> list[int]:
    n = len(ys)
    # x = a^(-1) mod P
    # x * a^(-1) = 1 mod P
    # 페르마의 소정리
    # a^(P - 1) = 1 mod P이므로 a^(P - 2) = a^(-1)
    n_inv = pow(n, p - 2, p)
    # w^(-i) = w^((N - i) % N)
    ws_inv = [0] * n
    ws_inv[0] = ws[0]
    for i in range(1, n):
        ws_inv[i] = ws[n - i]

    return [(n_inv * c) % p for c in fft(ys, ws_inv, p)]


# xs에서 다항식의 값을 계산하여 FFT 출력 확인
def eval_poly(f: list[int], xs: list[int], p: int) -> list[int]:
    ys = [0] * len(xs)
    for i, xi in enumerate(xs):
        x = 1
        y = 0
        for c in f:
            y += c * x
            x *= xi
        y %= p
        ys[i] = y

    return ys

In [2]:
P = 17
N = 16
g = 3

# 곱셈군
gs = [pow(g, i, P) for i in range(P - 1)]
assert len(set(gs)) == P - 1

# N차 원시 단위근
w = pow(g, (P - 1)//N, P)
print("w", w)
# w가 N차 원시 단위근인지 확인
assert pow(w, N, P) == 1
assert pow(w, N//2, P) == -1 % P

# FFT 평가 영역
ws = [pow(w, i, P) for i in range(N)]
# 다항식 예시
f = [i + 1 for i in range(N)]
print("f", f)
print("ws", ws)

e = eval_poly(f, ws, P)
r = fft_rec(f, ws, P)
l = fft(f, ws, P)
i = ifft(l, ws, P)

print("e", e)
print("r", r)
print("l", l)
print("i", i)
assert r == e
assert l == e
assert i == f

# 다항식의 차수보다 큰 영역에서 다항식의 값 계산
f = [i + 1 for i in range(N//2)] + ([0] * (N//2))
print("f", f)

ys = fft(f, ws, P)
print("ys", ys)

cs = ifft(ys, ws, P)
print("cs", cs)

assert cs == f

w 3
f [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
ws [1, 3, 9, 10, 13, 5, 15, 11, 16, 14, 8, 7, 4, 12, 2, 6]
e [0, 8, 2, 15, 7, 4, 6, 5, 9, 13, 12, 14, 11, 3, 16, 10]
r [0, 8, 2, 15, 7, 4, 6, 5, 9, 13, 12, 14, 11, 3, 16, 10]
l [0, 8, 2, 15, 7, 4, 6, 5, 9, 13, 12, 14, 11, 3, 16, 10]
i [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
f [1, 2, 3, 4, 5, 6, 7, 8, 0, 0, 0, 0, 0, 0, 0, 0]
ys [2, 5, 1, 9, 12, 13, 3, 5, 13, 0, 6, 11, 14, 8, 8, 8]
cs [1, 2, 3, 4, 5, 6, 7, 8, 0, 0, 0, 0, 0, 0, 0, 0]


In [3]:
# 비트 순서 반전
n = 8
# for i in range(N):
#     print(i)

c = 0
rev = 0
for i in range(n):
    print(i, rev)
    # 왼쪽에서 오른쪽으로 올림 처리
    mask = n >> 1
    while rev & mask:
        # 마스크가 1인 위치를 0으로 설정
        rev &= ~mask
        # 1을 오른쪽으로 이동
        mask >>= 1
    # 올림 처리 후 올바른 비트 위치에 1을 배치
    rev |= mask

0 0
1 4
2 2
3 6
4 1
5 5
6 3
7 7
